[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/26_lora.ipynb)

# 🟠 Medium: LoRA (Low-Rank Adaptation)

Implement **LoRA** — parameter-efficient fine-tuning for large models.

$$h = W_0 x + \frac{\alpha}{r} B A x$$

### Signature
```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Requirements
- `self.linear`: frozen `nn.Linear` (weight & bias `requires_grad=False`)
- `self.lora_A`: `nn.Parameter(rank, in_features)` — random init
- `self.lora_B`: `nn.Parameter(out_features, rank)` — **zero** init
- Scaling: `alpha / rank`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [17]:
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias = True).requires_grad_(False)
        dataA = torch.randn((rank, in_features)) * ((1 / in_features) ** 0.5)
        self.lora_A = nn.Parameter(dataA, requires_grad=True)
        dataB = torch.zeros((out_features, rank))
        self.lora_B = nn.Parameter(dataB, requires_grad=True)

        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        # base + lora
        base = self.linear(x)
        #lora = (self.alpha / self.rank) * x @ (self.lora_B @ self.lora_A).T inefficient, 2 * in_feat * out_feat * n_samples + 2 * rank * in_feat * out_feat FLOPS
        lora = (self.alpha / self.rank) * (x @ self.lora_A.T) @ self.lora_B.T #efficient, 2 * in_feat * rank * n_samples + 2 * rank * out_feat * n_samples = 2 * n_samples * rank (in_feat + out_feat) FLOPS
        return base + lora

In [18]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
print('Total:    ', sum(p.numel() for p in layer.parameters()))

Output: torch.Size([2, 8])
Trainable: 96
Total:     232


In [19]:
# ✅ SUBMIT
from torch_judge import check
check('lora')


🧪 Testing: LoRA (Low-Rank Adaptation) (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Base weights frozen (0.7ms)
  ✅ [2/5] LoRA parameter shapes (0.6ms)
  ✅ [3/5] B=0 means output equals base (6.4ms)
  ✅ [4/5] Only LoRA params get gradients (1.7ms)
  ✅ [5/5] Forward computation (2.7ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (12.2ms total)
  Progress saved. Run status() to see your dashboard.

